In [ ]:
from pathlib import Path
import pandas as pd
idx_slice = pd.IndexSlice
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
import plotly.graph_objects as go

import sys
sys.path.append('..')
from scripts.plot_helpers import (
    chdir_to_root_dir,
    read_stats_dict,
    read_csv_nafix,
    prepare_dataframe,
    nice_title,
    load_plot_config,
    get_scen_col_function,
    save_plotly_fig)


chdir_to_root_dir()

In [ ]:
yaml_config = load_plot_config()

RUN_NAME_PREFIX = yaml_config['run']['name_prefix']
SUMMARY_VERSION = yaml_config['run']['summary_version']
SDIR = Path.cwd() / "results" / f"{RUN_NAME_PREFIX}_summary_{SUMMARY_VERSION}"
SDIR.mkdir(exist_ok=True, parents=True)

COUNTRIES = yaml_config['data']['countries']
YEARS = yaml_config['data']['years']
INDEX_LEVELS_TO_DROP = yaml_config['data']['index_levels_to_drop']

IDX_GROUP = idx_slice[[RUN_NAME_PREFIX], :, COUNTRIES, YEARS]
IDX_GROUP_NAME = "".join(COUNTRIES) + "_MAIN"

print(f"Index Group Name: {IDX_GROUP_NAME}")
print(f"Index Group: {IDX_GROUP}")

# Scenario filtering
SCEN_FILTER = yaml_config['data']['scen_filter']
scen_order = yaml_config['data']['scen_order']

# Get scenario column function
scen_col_func_name = yaml_config['data']['scen_col_function']
set_scen_col = get_scen_col_function(scen_col_func_name)
print(f"Using scenario column function: {scen_col_func_name}")

# Plot dimensions
width = yaml_config['plot']['width']
heights = yaml_config['plot']['heights']

# Default figure kwargs
fig_kwargs = yaml_config['plot']['default_kwargs'].copy()
fig_kwargs['width'] = width
fig_kwargs['height'] = heights['medium']

print(f"\nConfiguration loaded successfully!")
print(f"Countries: {COUNTRIES}")
print(f"Years: {YEARS}")

In [ ]:
index_cols = [
    "run_name_prefix", "run_name", "country", "year", "simpl", "clusters", "ll", "opts", "sopts", "discountrate", "demand", "h2export"
 ]
mean_marginal_prices = read_csv_nafix(SDIR / "marginal_prices.csv", index_col=index_cols)
mean_marginal_prices = prepare_dataframe(mean_marginal_prices, IDX_GROUP, INDEX_LEVELS_TO_DROP, set_scen_col, drop_zero=False)
mean_marginal_prices = mean_marginal_prices.query("scen in @SCEN_FILTER") if SCEN_FILTER else mean_marginal_prices

In [ ]:
df = mean_marginal_prices.query("variable in ['H2 export bus']").copy()


In [ ]:
df

## Marginal prices | H2 export price

In [ ]:
idx = pd.IndexSlice

df = mean_marginal_prices[
    (mean_marginal_prices['year'].isin([2035, 2050])) & 
    (mean_marginal_prices['variable'] == 'H2 export bus')
].copy()


In [ ]:
def marginal_price_dumbbell_fig(df, year, data_start="0.1MtH2export", data_end="0.7MtH2export"):

    df = df.copy()
    df = df[df.year==year]
    countries = df["country"].unique()

    data = {"line_x": [], "line_y": [], "data_start": [], "data_end": [], "valid_countries": []}
    skipped_countries = []
    
    for country in countries:
        scen_start = f"{country}-{data_start}"
        scen_end = f"{country}-{data_end}"
        
        start_data = df.loc[(df.scen == scen_start) & (df.country == country)]["value"]
        end_data = df.loc[(df.scen == scen_end) & (df.country == country)]["value"]
        
        if len(start_data) > 0 and len(end_data) > 0:
            start_val = start_data.values[0]
            end_val = end_data.values[0]
            
            data["data_start"].append(start_val)
            data["data_end"].append(end_val)
            data["line_x"].extend([start_val, end_val, None])
            data["line_y"].extend([country, country, None])
            data["valid_countries"].append(country)
        else:
            missing = []
            if len(start_data) == 0:
                missing.append(data_start)
            if len(end_data) == 0:
                missing.append(data_end)
            skipped_countries.append(f"{country} (missing: {', '.join(missing)})")
    
    if skipped_countries:
        print(f"⚠️  Skipped {len(skipped_countries)} country(ies) due to missing scenario data:")
        for country_info in skipped_countries:
            print(f"   - {country_info}")

    fig = go.Figure(
        data=[
            go.Scatter(
                x=data["line_x"],
                y=data["line_y"],
                mode="lines",
                showlegend=False,
                marker=dict(color="grey")
            ),
            go.Scatter(
                x=data["data_start"],
                y=data["valid_countries"],
                mode="markers+text",
                name=data_start,
                text=[f"{v:.1f}" for v in data["data_start"]],
                textposition="top center",
                textfont=dict(size=11),
                marker=dict(color="#A6BCC9", size=13)
            ),
            go.Scatter(
                x=data["data_end"],
                y=data["valid_countries"],
                mode="markers+text",
                name=data_end,
                text=[f"{v:.1f}" for v in data["data_end"]],
                textposition="top center",
                textfont=dict(size=11),
                marker=dict(color="#179c7d", size=13)
            ),
        ]
    )

    fig.update_layout(
        title=dict(text=nice_title(f"Marginal price for H2 at export port in {year}", 
                                   "Per country and H2 export volume in €/MWh_H2_LHV")),
        width=800,
        height=max(500, len(data["valid_countries"]) * 40),
        legend_itemclick=False
    )

    return fig


In [ ]:
fig = marginal_price_dumbbell_fig(df, year=2050, data_start="0.7MtH2export", data_end="4.0MtH2export")

fig.show()

In [ ]:
idx = pd.IndexSlice

df = mean_marginal_prices[
    (mean_marginal_prices['year'] == 2035) & 
    (mean_marginal_prices['variable'] == 'H2 export bus')
].copy()

In [ ]:
fig = marginal_price_dumbbell_fig(df, year=2035, data_start="0.1MtH2export", data_end="0.7MtH2export")

output_path = SDIR / "marginal_prices_2035.csv"
df.to_csv(output_path)
print(f"Saved to: {output_path}")

fig.show()